<a href="https://colab.research.google.com/github/Joohhnnyyy/Pro/blob/main/NEW_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [77]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [78]:
df.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26,52,38,Sandy,Maize,37,0,0,Urea
1,29,52,45,Loamy,Sugarcane,12,0,36,DAP
2,34,65,62,Black,Cotton,7,9,30,14-35-14
3,32,62,34,Red,Tobacco,22,0,20,28-28
4,28,54,46,Clayey,Paddy,35,0,0,Urea


In [79]:
df = pd.read_csv("new dataset.csv")

In [80]:
X = df.drop('Fertilizer Name', axis=1).copy()
y = df['Fertilizer Name'].copy()

In [81]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [3,4])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [82]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=True, random_state=42)

In [83]:
noise_factor = 0.5
X_train_noisy = X_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train.shape)


In [84]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_noisy = sc.fit_transform(X_train_noisy)
X_test = sc.transform(X_test)

In [85]:
X_train_noisy[0]

array([-1.4355717 ,  0.06739571,  0.35462892,  0.4106759 ,  0.71358965,
        1.82301336, -0.03341217,  0.52439088, -0.04585197,  0.0261559 ,
       -0.41846782,  0.07214223,  1.06182145, -0.70258932, -0.17058586,
        1.81659896,  1.53026355,  1.38760959, -0.06194873, -0.71180068,
       -0.53578108,  1.22115933])

In [86]:
noise_factor_test = 0.3
X_test_noisy = X_test + noise_factor_test * np.random.normal(loc=0.0, scale=1.0, size=X_test.shape)

In [87]:
rand = ExtraTreesClassifier(random_state = 42)
rand.fit(X_train_noisy,y_train)

ExtraTreesClassifier(random_state=42)

In [88]:
pred_rand = rand.predict(X_test_noisy)

In [89]:

import numpy as np
import pandas as pd
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
warnings.filterwarnings("ignore")




params = {
    'n_estimators': [300, 400, 500],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 8]
}

grid_rand = GridSearchCV(rand, params, cv=3, verbose=3, n_jobs=-1)


grid_rand.fit(X_train_noisy, y_train)

pred_rand = grid_rand.predict(X_test_noisy)


accuracy = accuracy_score(y_test, pred_rand)
print(f"Accuracy: {accuracy * 100:.2f}%")

print(classification_report(y_test, pred_rand))

print('Best score:', grid_rand.best_score_)
print('Best params:', grid_rand.best_params_)


Fitting 3 folds for each of 27 candidates, totalling 81 fits
Accuracy: 97.30%
              precision    recall  f1-score   support

    10-26-26       0.93      0.97      0.95        73
    14-35-14       0.99      0.95      0.97       146
    17-17-17       0.94      0.94      0.94        63
       20-20       0.97      0.98      0.97       137
       28-28       0.97      0.96      0.96       167
         DAP       0.96      0.98      0.97       178
        Urea       1.00      1.00      1.00       236

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000

Best score: 1.0
Best params: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 300}


In [90]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

df = pd.read_csv("new dataset.csv")


def predict_fertilizer(Temperature, Humidity, Moisture, Soil_Type, Crop_Type, Nitrogen, Phosphorous, Potassium):
    user_input = pd.DataFrame({
        'Temparature': [Temperature],
        'Humidity ': [Humidity],
        'Moisture': [Moisture],
        'Soil Type': [Soil_Type],
        'Crop Type': [Crop_Type],
        'Nitrogen': [Nitrogen],
        'Phosphorous': [Phosphorous],
        'Potassium': [Potassium]
    })

    user_input_encoded = np.array(ct.transform(user_input))


    user_input_scaled = sc.transform(user_input_encoded)

    noise_factor_user = 0.3
    user_input_noisy = user_input_scaled + noise_factor_user * np.random.normal(loc=0.0, scale=1.0, size=user_input_scaled.shape)

    prediction = grid_rand.predict(user_input_noisy)

    return prediction[0]

temperature = 28
humidity = 54
moisture = 41
soil_type = "Clayey"
crop_type = "Paddy"
nitrogen = 36
phosphorous = 10
potassium = 10


predicted_fertilizer = predict_fertilizer(temperature, humidity, moisture, soil_type, crop_type, nitrogen, phosphorous, potassium)
print(f"The recommended fertilizer is: {predicted_fertilizer}")

The recommended fertilizer is: Urea
